In [ ]:
# 260911-------------------------------------------------------------------------------------------------------------------

In [ ]:
# 크롤링 코드

In [56]:
from selenium import webdriver # 크롤러 역할
from selenium.webdriver.common.by import By # 크롤러가 어떤 방식으로 값을 찾아오게 할 것인지 결정
from selenium.webdriver.chrome.service import Service # 크롤러에게 로컬컴퓨터 내 크롬브라우저 드라이브 일치 & 설치 지원
from selenium.webdriver.chrome.options import Options # 크롤러가 어떤 환경에서 어떤 방식으로 크롤링 하도록 할지 옵션 설정
from webdriver_manager.chrome import ChromeDriverManager # 실제 드라이브 설치 역할

import pandas as pd
import time # 특정 사이트 접속 후 스크립트 기반의 코드를 서버로부터 가져올 때까지 시간을 벌어주는 역할
import re # regular expression
service = Service(ChromeDriverManager().install())
options = Options()

# options.add_argument("--headless") # 가상브라우저를 현재 내 로컬컴퓨터에 나타나지 않도록 할 때
# options.add_argument("--disable-gpu") # 가상브라우저를 사용하지 않기 때문에 굳이 gpu 메모리를 사용하지 않겠다는 의미
options.add_argument("--disable-dev-shm-usage") # 큰 데이터의 크롤링 시, 데이터를 개발자도구 내 임시메모리 공간을 사용 x (로컬공간) 
options.add_argument("--window-size=1920,1080") # 크롤러의 웹 브라우저 사이즈 세팅
options.add_argument("--start-maximized") # 주변 환경요소와 무관하게 브라우저 사이즈를 무조건 크게
options.add_argument("--user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/152.0.0.0 Safari/537.36") # 브라우저 식별 정보 = 우리가 정상적인 웹사이트 방문자
options.add_argument("--lang=ko_KR") # 접속 및 방문하 브라우저의 기본세팅 언어
options.add_argument("--no-sandbox") # 규제 적용 방지

URL = "https://display.wconcept.co.kr/rn/women"

driver = webdriver.Chrome(service=service, options=options)
driver.get(URL)
time.sleep(3)

# 유틸리티 함수
def to_int(text) :
    num = re.sub(r"[^0-9]", "", text)
    return int(num) if num else 0

# 베스트 버튼 클릭
best_button = driver.find_element(By.XPATH, "//span[text()='베스트']")
best_button.click()
time.sleep(3)

# 상품카드 정보 크롤링
items = driver.find_elements(By.CSS_SELECTOR, "div.product-item.item.type-all")
print(f"1. ✅ 찾은 상품 수: {len(items)}개")

# 상품 수집데이터 저장용 리스트 생성
products = list() # []

# 상품별 상세데이터 수집
for idx, item in enumerate(items, 1) :
    # 상품 랭킹 번호 수집
    ranking_no = int(item.find_element(By.CSS_SELECTOR, "span.rank").text)

    # 브랜드 이름 수집
    brand_name = item.find_element(By.CSS_SELECTOR, "span.text.title").text.strip()

    # 상품 이름 수집
    product_name = item.find_element(By.CSS_SELECTOR, "span.text.detail").text.strip()
    
    # 판매가격 (*소비자가, 할인가, 할인률) 수집
    price_box = item.find_element(By.CSS_SELECTOR, "span.prdc-price")

    # 소비자가
    try :
        original_price = to_int(price_box.find_element(By.CSS_SELECTOR, "span.customer-price").text)
    except :
        original_price = 0

    # 할인가
    try :
        sale_price = to_int(price_box.find_element(By.CSS_SELECTOR, "span.final-price").text)
    except :
        sale_price = 0
        
    # 할인률
    try :
        discount_rate = to_int(price_box.find_element(By.CSS_SELECTOR, "span.final-discount").text)
    except :
        discount_rate = 0

    # 평점.리뷰수.좋아요수
    # try :
    #     stats = item.find_element(By.CSS_SELECTOR, "span.stats")
    
    #     # 평점
    #     try :
    #         rating = float(stats.find_element(By.CSS_SELECTOR, "span.review em.score").text)
    #     except :
    #         rating = 0.0
            
    #     # 리뷰수
    #     try :
    #         review_count = to_int(stats.find_element(By.CSS_SELECTOR, "span.review span.text.cnt").text)
    #     except :
    #         review_count = 0
            
    #     # 좋아요수
    #     try :
    #         like_count = to_int(stats.find_element(By.CSS_SELECTOR, "span.like span.text.cnt").text)
    #     except :
    #         like_count = 0
    # except :
    #     continue

    # 평점.리뷰수.좋아요수

    try :
        stats = item.find_element(By.CSS_SELECTOR, "span.stats")
    
        # 평점
        try :
            rating = float(stats.find_element(By.CSS_SELECTOR, "span.review em.score").text)
        except :
            rating = 0.0
    
        # 리뷰수
        try :
            review_count = to_int(stats.find_element(By.CSS_SELECTOR, "span.review span.text.cnt").text)
        except :
            review_count = 0
    
    
        # 좋아요수
        try :
            like_count = to_int(stats.find_element(By.CSS_SELECTOR, "span.like span.text.cnt").text)
        except :
            like_count = 0
    
    except :
        rating = 0.0
        review_count = 0
        like_count = 0

    # 상품상세페이지 url
    img_src = item.find_element(By.CSS_SELECTOR, "span.img img").get_attribute("src")

    match = re.search(r"/(\d{9})[_.]", img_src)

    if match :
        product_id = match.group(1)
        product_url = f"https://www.wconcept.co.kr/Product/{product_id}"
    else :
        product_id = None
        product_url = ""

    products.append({
        "ranking_no": ranking_no,
        "brand_name": brand_name,
        "product_name": product_name,
        "original_price": original_price,
        "sale_price": sale_price,
        "discount_rate": discount_rate,
        "rating": rating,
        "review_count": review_count,
        "like_count": like_count,
        "product_id": product_id,
        "product_url": product_url,
        "img_src": img_src
    })
    # https://www.wconcept.co.kr/Product/308577412
    # print(ranking_no, brand_name, product_name, original_price, sale_price, discount_rate, rating, review_count, like_count, product_id, product_url, img_src)
product_df= pd.DataFrame(products)

# driver.quit() # 특정 사이트 데이터 수집 후 가상 브라우저 실행 종료 // 메모리 사용 = 메모리 누수
print("2. ✅ wconcept 상품기본 정보수집 완료!")

# 각 상품별 리뷰수집 데이터 저장용 리스트
all_reviews = list() # []

for product in products[:3] :
    if product["review_count"] == 0 :
        print("리뷰없음")
        continue

    product_url = product["product_url"]

    # 상품 상세페이지 진입단계
    driver.get(product_url)

    time.sleep(3)

    try : 
        review_button = driver.find_element(By.CSS_SELECTOR, "#reviewCnt1")
        review_button.click()
    
        time.sleep(3)
        # print("2-1. ✅ 리뷰버튼 클릭 완료!")
    except :
        # print("2-2. ✅ 리뷰버튼이 없는 상품입니다 -> SKIP")
        continue

    print("3. ✅ wconcept 상품상세페이지 진입 완료!")
    for page_no in range(1, 4) :
        if page_no > 1 :
            page_buttons = driver.find_elements(By.CSS_SELECTOR, f"ul#reviewPageNavigation a[title='{page_no}']")

            if len(page_buttons) == 0 :
                print(f"{page_no}페이지가 없습니다.")
                print("리뷰 수집을 종료합니다.")

                break

            page_buttons[0].click()

            time.sleep(2)
            
        
        rows = driver.find_elements(By.CSS_SELECTOR, "tr")

        # 진짜 리뷰 정보값을 가지고 있는 tr만 저장할 목적의 리스트
        review_items = list()
    
        for row in rows :
            review_text_elements = row.find_elements(By.CSS_SELECTOR, "p.pdt_review_text")
    
            if len(review_text_elements) > 0 :
                review_items.append(row)

        print(f"{page_no}) 페이지 리뷰 수: {len(review_items)}개")
    
        for review_index, review in enumerate(review_items, 1) :

            review_no = (page_no - 1) * 6 + review_index
            
            # 리뷰 작성자
            reviewer = review.find_element(By.CSS_SELECTOR, "p.product_review_info_right > em").text.strip()
    
            # 리뷰 작성일
            review_date = review.find_element(By.CSS_SELECTOR, "p.product_review_info_right > span").text.strip()
    
            # 구매 옵션정보
            try :
                purchase_text = review.find_element(By.CSS_SELECTOR, "div.pdt_review_option span").text.strip()
                purchase_option = re.sub(r"^구매옵션\s*:\s*", "", purchase_text)
                purchase_option = purchase_option.rstrip(",").strip()
                
                if purchase_option.isdigit() :
                    purchase_option = int(purchase_option)
            except :
                purchase_option = ""
                
                # match = re.search(r"\d+", purchase_text) # group(0), (1)
                # purchase_option = int(match.group(0))
    
            # 리뷰 본문
            review_text = review.find_element(By.CSS_SELECTOR, "p.pdt_review_text").text.strip()
    
            # 리뷰 별점
            star = review.find_element(By.CSS_SELECTOR, "div.star-grade > strong")
            style = star.get_attribute("style")
            match = re.search(r"width:\s*([0-9]+)%", style)
    
            if match :
                star_percent = float(match.group(1))
                review_rating = star_percent / 20
            else :
                review_rating = 0.0
    
            # 리뷰 상세 평가 데이터 저장을 위한 딕셔너리 정의
            evaluations = dict() # | {}
    
            # 리뷰 상세 평가
            try :
                evaluation_items = review.find_elements(By.CSS_SELECTOR, "ul.product_review_evaluation > li")
                for evaluation in evaluation_items :
                    evaluation_type = evaluation.find_element(By.CSS_SELECTOR, "strong").text.strip()
                    evaluation_value = evaluation.find_element(By.CSS_SELECTOR, "em").text.strip()
                    evaluations[evaluation_type] = evaluation_value
            except :
                continue
    
            # 리뷰 이미지 저장용 리스트
            review_images = list()
    
            # 리뷰 이미지
            images = review.find_elements(By.CSS_SELECTOR, "ul.pdt_review_photo img")
        
            for image in images :
                image_url = image.get_attribute("src")
                review_images.append(image_url)

            review_data = {
                "product_id": product["product_id"],
                "brand_name": product["brand_name"],
                "product_name": product["product_name"],
                "product_url": product["product_url"],
                "review_page": review_page,
                "review_no": review_no,
                "reviewer": reviewer,
                "review_date": review_date,
                "purchase_option": purchase_option,
                "review_text": review_text,
                "review_rating": review_rating,
                "evaluations": evaluations,
                "review_images": review_images                
            }

            all_reviews.append(review_data)
            
        #     print(review_no, reviewer, review_date, purchase_option, review_text, review_rating, evaluations, review_images)
        # print(f"4. ✅ 리뷰 {page_no}페이지 수집 완료 -> {len(review_items)}개")
print("4. ✅ wconcept 전체리뷰수집 완료")

1. ✅ 찾은 상품 수: 200개
2. ✅ wconcept 상품기본 정보수집 완료!
리뷰없음
3. ✅ wconcept 상품상세페이지 진입 완료!
1) 페이지 리뷰 수: 6개
2) 페이지 리뷰 수: 6개
3) 페이지 리뷰 수: 6개
3. ✅ wconcept 상품상세페이지 진입 완료!
1) 페이지 리뷰 수: 6개
2) 페이지 리뷰 수: 6개
3) 페이지 리뷰 수: 6개
4. ✅ wconcept 전체리뷰수집 완료


In [57]:
brand_names = product_df["brand_name"].dropna().drop_duplicates().tolist()
print(f"{len(brand_names)}개")

95개


In [ ]:
# 네이버 블로그 API

In [58]:
import re
import requests
import pandas as pd

NAVER_CLIENT_ID = "5db4g311ij"
NAVER_CLIENT_SECRET = "vXbUcbv0h6CSpVSdjTVwrhYBK31SpCQ2zKgE4o9M"

NAVER_BLOG_API_URL = "https://naverapihub.apigw.ntruss.com/search/v1/blog"

headers = {
    "X-NCP-APIGW-API-KEY-ID": NAVER_CLIENT_ID,
    "X-NCP-APIGW-API-KEY": NAVER_CLIENT_SECRET
}

#유틸리티 함수
def clean_text(text) :
    text = re.sub(r"<[^>]+>", "", text)
    text = re.sub(r"&amp;", "", text)
    return text.strip()

brand_names = product_df["brand_name"].dropna().drop_duplicates().tolist()
brand_names = brand_names[:1]


# 블로그 컨텐츠 중 취합할 요소만 찾아서 컨텐츠별 저장할 딕셔너리를 모아놓기위한 용도의 리스트
all_blog_posts = list() # []

for brand_index, brand_name in enumerate(brand_names,  1) :
    params = {
        "query": brand_name,
        "display": 10,
        "start": 1,
        "sort": "date",
        "format": "json"
    }
    try :
        response = requests.get(NAVER_BLOG_API_URL, headers=headers, params=params, timeout = 10)

        if response.status_code == 200 :
            data = response.json()
            items = data.get("items", []) #대체 값으로 []가져와라

            for item in items :
                blog_data = {
                    "search_brand" : brand_name,
                    "title" : clean_text(item.get("title", "")), #대체값으로 ""가져와라
                    "link" : item.get("link", ""), 
                    "description" : clean_text(item.get("description", "")),
                    "bloggername" : item.get("bloggername", ""),
                    "bloggerlink" : item.get("bloggerlink", ""),
                    "postdate" : item.get("postdate", "")
                }
                
                all_blog_posts.append(blog_data)               
                
    except Exception as e :
        print(f"API 호출 중 오류 발생 : {e}") #오류의 구체적인 부분을 알려줌

blog_df = pd.DataFrame(all_blog_posts)

blog_df["postdate"] = pd.to_datetime(blog_df["postdate"], format="%Y%m%d", errors="coerce") #NaT = Not a Time
#만약, 1,000개의 데이터를 수집하고 싶다면? (100개씩 가져와서 총 1000개를 수집하는 반복문)
# for start in range(1, 1000, 100) :
#     params = {
#         "query": brand_name,
#         "display": 10,
#         "start": start,
#         "sort": "date",
#         "format": "json"
#     }

In [60]:
blog_df.limit(3)

,search_brand,title,link,description,bloggername,bloggerlink,postdate
0,팁토이조이,[아기용품] 걸음마 신발 아띠빠스 팁토이조이,https://blog.naver.com/be_happy3_/224406531359,사이즈 안내에 적힌 발사이즈로 맞춰서 주문해보았는데 (+ 더불어 팁토이조이사러 오프...,반짝반짝 빛나는,https://blog.naver.com/be_happy3_,2026-09-09
1,팁토이조이,아기 걷기를 도와주는 팁토이조이 추천,https://blog.naver.com/454y3zo4akge7/224405355720,아기 걷기를 도와주는 팁토이조이 추천 [팁토이조이] 아기 유아 걸음마신발 프리미엄 ...,데일리라이프,https://blog.naver.com/454y3zo4akge7,2026-09-08
2,팁토이조이,아기 걸음마 신발 뉴발란스 뉴비퍼스트 100 핑크 아이보리블루 구....,https://blog.naver.com/hahaohj/224404146313,아마 머지 않아 걸음마를 할 거라는 생각에 들뜨는 마음으로 아기 걸음마 신발을 사줬...,웰컴 우씨네 월드 ෆ˙ᵕ˙ෆ,https://blog.naver.com/hahaohj,2026-09-07
3,팁토이조이,아기 첫 걸음마 신발 추천 팁토이조이 내돈내산 솔직한 후기,https://blog.naver.com/dlcodms8593/224401300517,그렇게 선택했던 신발이 팁토이조이 신발이에요. 아이가 좋아해서 작은 사이즈도 신으려...,띠용이지유,https://blog.naver.com/dlcodms8593,2026-09-04
4,팁토이조이,아기 첫 신발 - 팁토이조이,https://blog.naver.com/kjh6680971/224399797493,팁토이조이 - 걸음마 신발 유후가 손을 잡지 않고 한발 두발 발을... 그러는 와중...,일상의 기록,https://blog.naver.com/kjh6680971,2026-09-03
5,팁토이조이,번쩍배송 팁토이조이 아기 신발 돌 걸음마 후기 참고,https://blog.naver.com/npmsmz4qaovmjk/22439868...,번쩍배송 팁토이조이 아기 신발 돌 걸음마 후기 참고 아기 신발을 찾던 중에 번쩍배송...,하루쭈봉,https://blog.naver.com/npmsmz4qaovmjk,2026-09-02
6,팁토이조이,뉴보시 팁토이조이 걸음마 운동화 돌선물 아기첫 유아신발 추천,https://blog.naver.com/iuzwvhnxbwmox/224397465631,그러다 발견한 게 바로 요 뉴보시 팁토이조이 걸음마 운동화예요.... 뉴보시 팁토이...,식탁 위의 기록,https://blog.naver.com/iuzwvhnxbwmox,2026-09-01
7,팁토이조이,17개월 아기 뉴발란스 IZ530KA 운동화 130 후기!,https://blog.naver.com/alldayon/224395226217,저희 아기는 오니츠카는 130 나이키도 130 팁토이조이는 126 신고 있어요. 사...,오니르니 리빙앤홈,https://blog.naver.com/alldayon,2026-08-30
8,팁토이조이,팁토이조이 펑키F 타피오카 돌 걸음마 신발 사이즈 선택법,https://blog.naver.com/eh92usy3ts3qppslxz6/224...,"팁토이조이 펑키F 타피오카를 살펴보며 가격과 옵션, 사이즈를 고를 때 체크할 기준을...",이히나인님의 블로그,https://blog.naver.com/eh92usy3ts3qppslxz6,2026-08-28
9,팁토이조이,[아기 첫 신발 추천 ]-팁토이조이 126사이즈 내돈내산 후기,https://blog.naver.com/mmon3333/224392450311,오늘은 첫 걸음마 신발로 유명한 팁토이조이 신발 내돈내산 후기... 팁토이조이 걸음...,여노의 일상,https://blog.naver.com/mmon3333,2026-08-27


In [ ]:
# 아래부터는 위에서 진행했던 크롤링의 값을 csv 파일로 저장했고, 그 파일 내 값을 SQL로 보내는 과정
# 크롤링 완성 파일은 세진샘 제공(products, reviews, blogs)

In [1]:
import pandas as pd

product_df = pd.read_csv("./Wconcept_data/wconcept_products_20260913_234742.csv") #.은 루트폴더 /는 하위 경로로 들어가는 것을 의미함
review_df = pd.read_csv("./Wconcept_data/wconcept_reviews_20260913_234742.csv")
blog_df = pd.read_csv("./Wconcept_data/wconcept_blogs_20260914_001928.csv")

In [2]:
!pip install mysql-connector-python

In [2]:
import mysql.connector

conn = mysql.connector.connect(
    host="localhost",
    user="root", 
    password="ssojini9804",
    database="wconcept_db_260917_01"
)

cursor = conn.cursor()

In [ ]:
# 실행횟수 초기화 하는 코드
conn.rollback() #commit 하지 않은 INSERT 작업(=임시메모리 공간에 있었던 작업)은 모두 취소

cursor.execute("SET FOREIGN_KEY_CHECKS = 0")
cursor.execute("TRUNCATE TABLE crawl_runs")
cursor.execute("SET FOREIGN_KEY_CHECKS = 1")

print("crawl_runs 초기화")

In [3]:
# 파이썬을 통해 수집한 데이터의 결측치의 값인 unll을 NaN, NaT로 취급. 
# 하지만, mysql에서는 읽을 수 없음 -> None의 형태로 변환시켜야 함
import pandas as pd
from datetime import datetime

def to_none(value) :
    if pd.isna(value) :
        return None
    return value

def to_date(value) :
    if isinstance(value, pd.Timestamp) :
        return value.date()
    return value

In [4]:
collected_at = datetime.now()

sql = """
INSERT INTO crawl_runs (collected_at) VALUES (%s)
"""

cursor.execute(sql, (collected_at, ))

crawl_run_id = cursor.lastrowid

print("crawl_run_id:", crawl_run_id)

crawl_run_id: 4


In [5]:
brand_names = (
    product_df["brand_name"]
        .dropna()
        .drop_duplicates()
        .tolist()
)

In [9]:
brand_names

['틸아이다이',
 '드로우핏 우먼',
 '마인드브릿지 우먼',
 '아비스파',
 '아워코모스',
 '우이',
 '어스',
 '아디다스',
 '로브로브',
 '하우투러브미',
 '룩캐스트',
 '반원 아틀리에',
 '망고매니플리즈',
 '루에브르',
 '지지피엑스',
 '마르벳',
 '로트르드민',
 '오스트카카',
 '엽페',
 '라운드트리코',
 '모노로우',
 '무니드',
 '파사드패턴',
 '프론트로우',
 '르세지엠',
 '엔젤삭스',
 '자빈드서울',
 '더라우스트',
 '엘리자베스 스튜어트',
 '메무아',
 '꼼파뇨 키즈',
 '시티브리즈',
 '뉴발란스',
 '슬로우롤리',
 '바시카',
 '논앤논',
 '시야쥬',
 '수아레 우먼',
 '모한',
 '프렌치시크',
 '에임에이지 스튜디오',
 '언에디트',
 '세이지먼트',
 '앤하츠',
 '레브바이레브',
 '에토스',
 '씨씨콜렉트',
 '쎄씨',
 '르메메',
 '유메르',
 '제이청',
 '플로움',
 '마롭',
 '로우틸',
 '베르네프',
 '아틀리에 나인',
 '레테라',
 '올리브데올리브',
 '오디에르',
 '레이첼콕스',
 '사르투',
 '듀엘',
 '팁토이조이',
 '휠라',
 '드메리엘',
 '오프 앤드 어웨이',
 '문트',
 '비에이유 바이 브라이드앤유',
 '잇미샤',
 '마스 언 에브릴',
 '미드가르드',
 '아티드',
 '오브에르',
 '그레이데시온',
 '헌치',
 '아티카',
 '노노어',
 '프론투라인',
 '히비스커스블룸',
 '뮤니에',
 '주르티',
 '리드볼트',
 '토와브',
 '스퍼',
 '비올셋',
 '라메레이',
 '어나우트',
 '루시다',
 '시엔느']

In [6]:
sql = """
INSERT INTO brands (brand_name) VALUES (%s) 
ON DUPLICATE KEY UPDATE brand_name = VALUES(brand_name)
"""

In [7]:
for brand_name in brand_names :
    cursor.execute(sql, (brand_name, ))

In [8]:
sql = """
SELECT brand_id, brand_name FROM brands
"""

In [9]:
cursor.execute(sql)

brand_rows = cursor.fetchall() # [(1, "adidas"), (2, "nike"), ...]

In [10]:
# dictionary / listcomprehension 문법명칭/용어 -> 딕셔너리 /리스트 형태의 자료구조를 생성, 손쉽게 만들어주는 문법
# tuple 구조분해할당 문법
brand_map = {brand_name: brand_id for brand_id, brand_name in brand_rows}

In [38]:
product_sql = """
INSERT INTO products (
    product_id,
    brand_id,
    product_name,
    product_url,
    img_url,
    category_depth1,
    category_depth2,
    category_depth3,
    category_depth4
)
VALUES(
    %s, %s, %s, %s, %s,
    %s, %s, %s, %s
)
ON DUPLICATE KEY UPDATE
    brand_id = VALUES(brand_id),
    product_name = VALUES(product_name),
    product_url = VALUES(product_url),
    img_url = VALUES(img_url),
    category_depth1 = VALUES(category_depth1),
    category_depth2 = VALUES(category_depth2),
    category_depth3 = VALUES(category_depth3),
    category_depth4 = VALUES(category_depth4)
"""

In [39]:
for _, row in product_df.iterrows() : #iterrows() -> df 안의 값을 행단위로 조회, 해당 행이 몇번째 인덱스 인지(인덱스가 필요 없으니까 _로 표기), 행 전체 값
    product_id = str(row["product_id"])
    brand_id = brand_map.get(row["brand_name"])

    cursor.execute(
        product_sql,
        (
            product_id,
            brand_id,
            to_none(row["product_name"]),
            to_none(row["product_url"]),
            to_none(row["img_url"]),
            to_none(row["category_depth1"]),
            to_none(row["category_depth2"]),
            to_none(row["category_depth3"]),
            to_none(row["category_depth4"])
        )
    )

In [40]:
snapshot_sql = """
INSERT INTO product_snapshots (
    crawl_run_id,
    product_id,
    original_price,
    sale_price,
    discount_rate,
    rating,
    review_count,
    like_count
)
VALUES(
    %s, %s, %s, %s,
    %s, %s, %s, %s
)
ON DUPLICATE KEY UPDATE
    original_price = VALUES(original_price),
    sale_price = VALUES(sale_price),
    discount_rate = VALUES(discount_rate),
    rating = VALUES(rating),
    review_count = VALUES(review_count),
    like_count = VALUES(like_count)
"""

In [41]:
for _, row in product_df.iterrows() : 
    cursor.execute(
        snapshot_sql,
        (
            crawl_run_id,
            str(row["product_id"]),
            to_none(row["original_price"]),
            to_none(row["sale_price"]),
            to_none(row["discount_rate"]),
            to_none(row["rating"]),
            to_none(row["review_count"]),
            to_none(row["like_count"])
        )
    )

In [42]:
ranking_sql = """
INSERT INTO product_rankings (
    crawl_run_id,
    product_id,
    ranking_no
)
VALUES (
    %s, %s, %s
)
ON DUPLICATE KEY UPDATE
    ranking_no = VALUES(ranking_no)
"""

In [43]:
for _, row in product_df.iterrows() :
    cursor.execute(
        ranking_sql,
        (
            crawl_run_id,
            str(row["product_id"]),
            int(row["ranking_no"])
        )
    )

In [44]:
review_sql = """
INSERT INTO reviews (
    crawl_run_id,
    product_id,
    review_page,
    review_no,
    reviewer,
    review_date,
    purchase_option,
    review_text,
    review_rating
)
VALUES (
    %s, %s, %s, %s, %s, 
    %s, %s, %s, %s 
)
ON DUPLICATE KEY UPDATE
    review_id= LAST_INSERT_ID(review_id),
    review_page= VALUES(review_page),
    review_no= VALUES(review_no),
    reviewer= VALUES(reviewer),
    review_date= VALUES(review_date),
    purchase_option= VALUES(purchase_option),
    review_text= VALUES(review_text),
    review_rating= VALUES(review_rating),
"""

In [48]:
import ast
# ast = Abstract Syntax Tree = 추상 구문 트리
# Abstract = 추상화

review_df["evaluations"] = review_df["evaluations"].apply(
    lambda x: ast.literal_eval(x)
    if pd.notna(x)
    else {}
)

review_df["review_images"] = review_df["review_images"].apply(
    lambda x: ast.literal_eval(x)
    if pd.notna(x)
    else []
)

ValueError: malformed node or string: {'사이즈': '적당함', '색상': '같음', '소재': '같음'}

In [49]:
evaluation_sql = """
INSERT INTO review_evaluations (
    review_id,
    evaluation_type,
    evaluation_value
)
VALUES(
    %s, %s, %s
)
ON DUPLICATE KEY UPDATE
    evaluation_value= VALUES(evaluation_value)
"""

In [25]:
image_sql = """
INSERT INTO review_images (
    review_id,
    image_url
)
VALUES(
    %s, %s
)
ON DUPLICATE KEY UPDATE
    image_url = VALUES(image_url)
"""

In [33]:
import pandas as pd

for _, row in review_df.iterrows() :
    cursor.execute(
        review_sql,
        (
            crawl_run_id,
            str(row["product_id"]),
            int(row["review_page"]),
            int(row["review_no"]),
            str(row["reviewer"]),
            to_date(row["review_date"]),
            str(row["purchase_option"]) if pd.notna(row["purchase_option"]) else None,
            to_none(row["review_text"]),
            to_none(row["review_rating"])
        )
    )
    review_id = cursor.lastrowid

    evaluations = row["evaluations"]

    if isinstance(evaluations, dict) :
        for evaluation_type, evaluation_value in evaluations.items() : 
            cursor.execute(
                evaluation_sql,
                (
                    review_id,
                    evaluation_type,
                    evaluation_value
                )
            )
            
    review_images = row["review_images"]

    if isinstance(review_images, list) :
        for image_url in review_images :
            if not review_url :
                continue

            cursor.execute(
                image_sql,
                (
                    review_id,
                    image_url
                )
            )
    

ProgrammingError: 1064 (42000): You have an error in your SQL syntax; check the manual that corresponds to your MySQL server version for the right syntax to use near '' at line 24

In [ ]:
blog_sql = """
INSERT INTO blog_posts(
    crawl_run_id,
    brand_id,
    search_brand,
    title,
    link,
    description,
    blogername,
    bloggerlink,
    postdate
)
VALUES (
    %s, %s, %s, %s, %s, 
    %s, %s, %s, %s
)
ON DUPLICATE KEY UPDATE
    title = VALUES(title),
    link = VALUES(link),
    description = VALUES(description),
    blogername = VALUES(blogername),
    bloggerlink = VALUES(bloggerlink),
    postdate = VALUES(postdate)
"""

In [ ]:
for _, row in blog_df.interrows() :
    brand_id = brand_map.get(
        row["search_brand"]
    )

    cursor.execute(
        blog_sql,
        (
            crawl_run_id,
            brand_id,
            row["search_brand"],
            to_none(row["title"]),
            to_none(row["link"]),
            to_none(row["description"]),
            to_none(row["blogername"]),
            to_none(row["bloggerlink"]),
            to_date(row["postdate"])
        )
    )

In [ ]:
try :
    conn.commit()
    print("전체 데이터 저장 완료")
except Exception as e :
    conn.rollback()
    print("오류 발생 : ", e)
finally : 
    cursor.close()
    conn.close()